In [2]:
import numpy as np
import requests
import matplotlib.pyplot as plt

# pytorch stuff
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.dataset import random_split
import torch
import torch.nn as nn
from torch.nn import functional as F

# vector plots
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [3]:
# GPT-2's tokenizer
from transformers import GPT2LMHeadModel, GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

/home/robertcowher/anaconda3/envs/llm_course/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# data hyperparameters
n_vocab = 50257
embed_dim = 768 # 64
seq_len = 256 # context length

# model hyperparameters
n_heads = 12
n_blocks = 12 

# training hyperparameters
batch_size = 16 


In [17]:
# tokenize the text and make it a tensor
# text = requests.get('https://www.gutenberg.org/files/35/35-0.txt').text
text = requests.get('https://www.gutenberg.org/cache/epub/65473/pg65473.txt').text
all_tokens = torch.tensor( tokenizer.encode(text) )
len(all_tokens)

154729

In [18]:
train_ratio = 0.9

test_split_point = int(train_ratio*len(all_tokens))

train_data = all_tokens[:test_split_point]
test_data = all_tokens[test_split_point:]

print(len(train_data))
print(len(test_data))

139256
15473


In [ ]:
def get_data_batch(training=True):

    if training:
        data = train_data
    else:
        data = test_data

    # This is wrong because it leaves out sequences
    # ix = np.random.randint(0, len(data) - 1, batch_size, dtype=int)

    # X = data[ix] 
    # y = data[ix + 1]

    ix = torch.randint(len(data)-seq_len, size=(batch_size,))

    X = all_tokens[ix[:,None] + torch.arange(seq_len)]
    y = all_tokens[ix[:,None] + torch.arange(1,seq_len+1)]

    return X, y

X, y = get_data_batch()

print(X.shape)

(tensor([[ 1231,   616,  8281,  ...,   391,   201,   198],
         [16337,    11,   290,  ...,   324, 10793,   282],
         [  284,   674, 11368,  ...,   220,   220,   220],
         ...,
         [  201,   198,   273,  ...,  6515,   612,   373],
         [ 2011,  4958,   550,  ...,   262,  4252,    13],
         [40867,   373,   284,  ...,   286,  8617,    13]]),
 tensor([[  616,  8281,   290,  ...,   201,   198,   220],
         [   11,   290,    11,  ..., 10793,   282,     8],
         [  674, 11368,    11,  ...,   220,   220,   465],
         ...,
         [  198,   273,  1352,  ...,   612,   373,   262],
         [ 4958,   550,   201,  ...,  4252,    13,   314],
         [  373,   284,  3151,  ...,  8617,    13,   383]]))